# Poster2Movie — 00. Construir el dataset compacto

Se corre **una sola vez** (lo corre quien arma el sistema base, no cada equipo).

Junta el dataset base de Hugging Face (`skvarre/movie_posters-100k`, 95,300 filas, 43 GB) con las películas de los equipos y deja en tu Google Drive (`Mi unidad/poster2movie_datos/`) un dataset de ~3.3 GB:

- sin películas repetidas (el base trae la misma película hasta 10 veces);
- con los 17 géneros de los equipos (sin Music ni TV Movie);
- solo películas con sinopsis de al menos 20 palabras;
- carteles reducidos a 384 px de ancho;
- separado en train / val / test **por franquicia**.

**No necesita GPU**: deja el entorno en CPU para no gastar tu cuota de GPU. Tarda ~1-1.5 h. Si Colab se desconecta, vuelve a ejecutar todo: continúa desde el último archivo terminado.

## 1. Guardar en Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DESTINO = "/content/drive/MyDrive/poster2movie_datos"

## 2. Código del proyecto

In [ ]:
REPO = "AndreaHortaHernandez/poster2movie"

import os
if not os.path.isdir("/content/repo"):
    !git clone https://github.com/{REPO}.git /content/repo
else:
    !cd /content/repo && git pull
%cd /content/repo
!pip install -q -r requirements.txt

## 3. Construir

Imprime una línea por cada uno de los 88 archivos del dataset base.

In [ ]:
!python -m p2m.construir_dataset {DESTINO}

## 4. Revisar el resultado

In [ ]:
import json, io
import pandas as pd
import pyarrow.parquet as pq
from PIL import Image
from IPython.display import display

resumen = json.load(open(f"{DESTINO}/resumen.json"))
print(json.dumps({k: v for k, v in resumen.items() if k != "por_genero"}, indent=1))
display(pd.DataFrame(resumen["por_genero"]).fillna(0).astype(int).sort_values("train", ascending=False))

# Unos carteles de ejemplo
muestra = pq.read_table(f"{DESTINO}/equipos.parquet").slice(0, 4).to_pylist()
for f in muestra:
    print(f["title"], f["generos"], f["split"])
    display(Image.open(io.BytesIO(f["imagen"])).resize((192, 288)))